# 🏥 HỆ THỐNG AI ĐA PHƯƠNG THÁI (2D + 3D CEUS) CHẨN ĐOÁN U BUỒNG TRỨNG
### *Multi-Modal Two-Stage Pipeline: YOLO11 Binary + EfficientNet-B3 (Focal Loss, 15% Padding, Macro-F1 & Grad-CAM XAI)*
---
**Tổng quan Đề tài Nghiên cứu Khoa học:**
- Hợp nhất **cả 2 bộ dữ liệu siêu âm MMOTU**: `OTU_2d` (1.469 ảnh) và `OTU_3d` (170 ảnh CEUS cản âm vi mạch) $\rightarrow$ Tổng cộng: **1.639 ảnh**.
- **Giai đoạn 1 (YOLO11 Binary Detector)**: Sàng lọc đầu vào **Có bệnh (Tumor) vs Bình thường (Normal Ovary)** và định vị hộp bao tổn thương (Bounding Box).
- **Giai đoạn 2 (EfficientNet-B3 Phân loại Bệnh học 7 Lớp)**:
  - Nhận ảnh cắt vi thể khối u kèm **15% padding bối cảnh biên** ($300 \times 300$ px).
  - Huấn luyện với **Focal Loss ($\gamma=2.0$, $\alpha$)** giải quyết triệt để mất cân bằng mẫu hiếm (*Theca cell tumor*, *High-grade serous*).
  - Tiêu chuẩn lưu checkpoint theo **Val Macro-F1 cao nhất** chống sụp đổ nhãn (*mode collapse*).
- **Explainable AI (Grad-CAM)**: Trích xuất bản đồ kích hoạt gradient để kiểm chứng mô hình tập trung vào vách ngăn, chồi thịt, bờ xâm lấn hay bị nhiễu bởi vệt thước đo/chữ máy siêu âm.
- **Báo cáo Đa phương thái**: Bảng thống kê độ nhạy/đặc hiệu trên 2D, 3D CEUS và Ma trận nhầm lẫn chuẩn hóa $7 \times 7$ sạch bóng nhãn background.

## 1. Thiết lập Môi trường & Kiểm tra Phần cứng GPU (CUDA)
> **Lưu ý:** Chọn menu **Runtime > Change runtime type > T4 GPU** trước khi bắt đầu.

In [ ]:
!pip install -q ultralytics torchvision timm opencv-python-headless pandas matplotlib seaborn tabulate scikit-learn

import torch
print("=" * 65)
print("🎮 KIỂM TRA PHẦN CỨNG GOOGLE COLAB")
print("=" * 65)
cuda_ok = torch.cuda.is_available()
print(f"CUDA Sẵn sàng: {cuda_ok}")
if cuda_ok:
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Dung lượng VRAM: {vram:.2f} GB")
else:
    print("⚠️ Cảnh báo: Vui lòng vào Runtime > Change runtime type > T4 GPU!")
print("=" * 65)

## 2. Kết nối Google Drive & Giải nén Dữ liệu MMOTU (2D & 3D)
Nén thư mục `MMOTU` (chứa `OTU_2d` và `OTU_3d`) thành file `MMOTU.zip` và kéo vào Google Drive (`MyDrive/MMOTU.zip`).

In [ ]:
# ==========================================================================
# 1. KẾT NỐI GOOGLE DRIVE (HOẶC TẢI TỰ ĐỘNG TỪ GITHUB NẾU KHÔNG DÙNG DRIVE)
# ==========================================================================
from google.colab import drive
import zipfile, os, shutil
from pathlib import Path

drive_mounted = False
try:
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    drive_mounted = True
    print('✅ Đã kết nối thành công Google Drive!')
except Exception as e:
    print(f'⚠️ Chưa thể kết nối Google Drive ({e}).')
    print('💡 Mẹo: Bạn có thể bấm biểu tượng Thư mục bên tay trái -> Bấm biểu tượng Google Drive để kết nối.')

extract_dest = Path('/content/dataset_raw')
extract_dest.mkdir(parents=True, exist_ok=True)

# 2. Kiểm tra dữ liệu trên Google Drive (nếu đã kết nối Drive)
found_drive_folder = None
if drive_mounted:
    drive_candidates = [
        Path('/content/drive/MyDrive/dataset/MMOTU'),
        Path('/content/drive/MyDrive/MMOTU'),
        Path('/content/drive/MyDrive/dataset'),
    ]
    for p in drive_candidates:
        if p.exists() and ((p / 'OTU_2d').exists() or list(p.glob('**/OTU_2d'))):
            found_drive_folder = p
            break

if found_drive_folder:
    print(f'🎯 Đã tìm thấy dataset trên Google Drive tại: {found_drive_folder}')
    target_2d = extract_dest / 'OTU_2d'
    target_3d = extract_dest / 'OTU_3d'
    if not target_2d.exists():
        print('⚡ Đang sao chép ảnh sang ổ SSD của Colab để huấn luyện GPU với tốc độ tối đa...')
        src_2d = list(found_drive_folder.glob('**/OTU_2d'))[0]
        shutil.copytree(src_2d, target_2d, dirs_exist_ok=True)
        src_3d_list = list(found_drive_folder.glob('**/OTU_3d'))
        if src_3d_list:
            shutil.copytree(src_3d_list[0], target_3d, dirs_exist_ok=True)
    print('✅ Dữ liệu MMOTU đã sẵn sàng tại /content/dataset_raw!')

elif drive_mounted and os.path.exists('/content/drive/MyDrive/MMOTU.zip'):
    print('📦 Tìm thấy file MMOTU.zip trên Drive. Đang giải nén...')
    with zipfile.ZipFile('/content/drive/MyDrive/MMOTU.zip', 'r') as zip_ref:
        zip_ref.extractall(extract_dest)
    print('✅ Giải nén thành công!')

else:
    print('🚀 Đang tự động nạp dữ liệu trực tiếp từ GitHub repository...')
    !git clone https://github.com/son7nui/NCKH-U-buong-trung.git /content/NCKH
    if (Path('/content/NCKH/MMOTU/OTU_2d')).exists():
        shutil.copytree('/content/NCKH/MMOTU/OTU_2d', extract_dest / 'OTU_2d', dirs_exist_ok=True)
        shutil.copytree('/content/NCKH/MMOTU/OTU_3d', extract_dest / 'OTU_3d', dirs_exist_ok=True)
    print('✅ Đã nạp thành công dữ liệu từ GitHub!')


## 3. Tiền xử lý Đa phương thái: Tự động Ghép Nối 2D + 3D CEUS & Cắt ROI Padding 15%
- Ghép nối 1.469 ảnh 2D và 170 ảnh 3D CEUS.
- Tạo tập Giai đoạn 1: Nhị phân (Có U vs Buồng trứng bình thường).
- Tạo tập Giai đoạn 2: Cắt ROI khối u với **15% padding** (lưu giữ vỏ bao, bờ ranh giới mô lành), **chỉ gồm đúng 7 nhóm bệnh học** (loại bỏ hoàn toàn Normal ovary và Background).

In [ ]:
import cv2
import numpy as np
import os, shutil
from pathlib import Path

base_raw = Path('/content/dataset_raw')

# Tìm thư mục OTU_2d và OTU_3d
dirs_2d = list(base_raw.glob('**/OTU_2d'))
dirs_3d = list(base_raw.glob('**/OTU_3d'))

otu_2d = dirs_2d[0] if dirs_2d else base_raw / 'OTU_2d'
otu_3d = dirs_3d[0] if dirs_3d else base_raw / 'OTU_3d'

print(f"Thư mục 2D: {otu_2d}")
print(f"Thư mục 3D: {otu_3d}")

OUTPUT_BINARY = Path('/content/dataset_multimodal_binary')
OUTPUT_CROPS = Path('/content/dataset_multimodal_crops_7cls')

TUMOR_7_CLASSES = {
    0: 'Chocolate_cyst',
    1: 'Serous_cystadenoma',
    2: 'Teratoma',
    3: 'Theca_cell_tumor',
    4: 'Simple_cyst',
    5: 'Mucinous_cystadenoma',
    6: 'High_grade_serous'
}

ORIGINAL_TO_7CLS = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 6: 5, 7: 6}

def load_cls(fpath):
    m = {}
    if not os.path.exists(fpath): return m
    with open(fpath, 'r', encoding='utf-8') as f:
        for l in f:
            p = l.strip().split()
            if len(p) >= 2: m[p[0]] = int(p[1])
    return m

def extract_boxes(mask_path, img_w, img_h, min_area=30):
    if not os.path.exists(mask_path): return [], []
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None: return [], []
    thresh = (mask > 0).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    yolo_boxes, pix_boxes = [], []
    for c in cnts:
        if cv2.contourArea(c) < min_area: continue
        x, y, w, h = cv2.boundingRect(c)
        pix_boxes.append((max(0, x), max(0, y), min(img_w, x + w), min(img_h, y + h)))
        xc = min(1.0, max(0.0, (x + w / 2.0) / img_w))
        yc = min(1.0, max(0.0, (y + h / 2.0) / img_h))
        nw = min(1.0, max(0.0, w / float(img_w)))
        nh = min(1.0, max(0.0, h / float(img_h)))
        yolo_boxes.append((xc, yc, nw, nh))
    return yolo_boxes, pix_boxes

def crop_roi(img, box, pad=0.15, sz=(300, 300)):
    """Cắt ROI với 15% biên mô đệm xung quanh (lưu giữ vỏ bao và bờ ranh giới âm học)."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    cx1 = max(0, x1 - int(bw * pad))
    cy1 = max(0, y1 - int(bh * pad))
    cx2 = min(w, x2 + int(bw * pad))
    cy2 = min(h, y2 + int(bh * pad))
    crop = img[cy1:cy2, cx1:cx2]
    if crop.size == 0: return None
    return cv2.resize(crop, sz, interpolation=cv2.INTER_LINEAR)

def ingest_modality(data_dir, prefix, split, stats):
    cls_map = load_cls(data_dir / f"{split}_cls.txt")
    imgs_dir = data_dir / 'images'
    annots_dir = data_dir / 'annotations'
    bin_img = OUTPUT_BINARY / 'images' / split
    bin_lbl = OUTPUT_BINARY / 'labels' / split
    os.makedirs(bin_img, exist_ok=True)
    os.makedirs(bin_lbl, exist_ok=True)
    
    for img_name, orig_cls in cls_map.items():
        base_id = Path(img_name).stem
        src = imgs_dir / img_name
        if not src.exists(): continue
        img = cv2.imread(str(src))
        if img is None: continue
        h, w = img.shape[:2]
        
        m_path = annots_dir / f"{base_id}_binary.PNG"
        if not m_path.exists(): m_path = annots_dir / f"{base_id}.PNG"
        y_boxes, p_boxes = extract_boxes(m_path, w, h)
        
        # 1. Binary YOLO dataset (Sàng lọc Có U vs Bình Thường)
        u_name = f"{prefix}_{base_id}"
        shutil.copy2(src, bin_img / f"{u_name}.JPG")
        with open(bin_lbl / f"{u_name}.txt", 'w', encoding='utf-8') as lf:
            if orig_cls == 5:
                stats['normal'] += 1
            else:
                for xc, yc, bw, bh in y_boxes:
                    lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
                stats['tumor'] += 1
                
        # 2. Crops 7 classes dataset (CHỈ DÀNH CHO 7 NHÓM BỆNH HỌC - KHÔNG CÓ NORMAL VÀ BACKGROUND)
        if orig_cls != 5 and orig_cls in ORIGINAL_TO_7CLS:
            c_name = TUMOR_7_CLASSES[ORIGINAL_TO_7CLS[orig_cls]]
            c_dir = OUTPUT_CROPS / split / c_name
            os.makedirs(c_dir, exist_ok=True)
            if p_boxes:
                for idx, pbox in enumerate(p_boxes):
                    roi = crop_roi(img, pbox, pad=0.15)
                    if roi is not None:
                        cv2.imwrite(str(c_dir / f"{u_name}_crop{idx}.jpg"), roi)
                        stats['crops'] += 1
            else:
                w_roi = cv2.resize(img, (300, 300))
                cv2.imwrite(str(c_dir / f"{u_name}_whole.jpg"), w_roi)
                stats['crops'] += 1

for s in ['train', 'val']:
    st = {'tumor': 0, 'normal': 0, 'crops': 0}
    ingest_modality(otu_2d, '2d', s, st)
    ingest_modality(otu_3d, '3d', s, st)
    print(f"[{s.upper()}] Có u: {st['tumor']} | Bình thường: {st['normal']} | Crops 7 lớp u: {st['crops']}")

# Tạo dataset.yaml cho YOLO11
with open(OUTPUT_BINARY / 'dataset.yaml', 'w') as yf:
    yf.write(f"path: {OUTPUT_BINARY}\ntrain: images/train\nval: images/val\nnames:\n  0: Tumor\n")
print("✅ Chuẩn hóa thành công bộ dữ liệu Đa phương thái với 15% ROI Padding!")

## 4. Huấn luyện Giai đoạn 1: YOLO11n Binary Detector
Sàng lọc đầu vào Có bệnh vs Bình thường và định vị hộp bao u trên 1.639 ảnh.

In [ ]:
from ultralytics import YOLO

model_stage1 = YOLO('yolo11n.pt')
print("Huấn luyện YOLO11n Binary trên GPU T4 (30 epochs)...")

res_yolo = model_stage1.train(
    data='/content/dataset_multimodal_binary/dataset.yaml',
    epochs=30,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    patience=10,
    project='/content/runs_multimodal',
    name='stage1_yolo11_binary',
    plots=True,
    verbose=True
)
print("✅ Giai đoạn 1 hoàn tất! Lưu tại: /content/runs_multimodal/stage1_yolo11_binary/weights/best.pt")

## 5. Huấn luyện Giai đoạn 2: EfficientNet-B3 với Focal Loss ($\gamma=2.0$, $\alpha$) & Tối ưu Macro-F1
- **Mục tiêu**: Nhận diện vi cấu trúc phản âm u $300 \times 300$ px (kèm 15% padding).
- **Hàm mất mát**: **Focal Loss ($\gamma=2.0$)** kết hợp trọng số nghịch đảo $\alpha$ (`class_weights`) triệt tiêu gradient mẫu dễ, ép mô hình học chuẩn xác nhóm ca hiếm/ác tính (*Theca cell tumor*, *High-grade serous*).
- **Tiêu chí chọn Checkpoint**: Lưu mô hình có **Val Macro-F1 cao nhất** (bảo vệ các lớp thiểu số, ngăn chặn tuyệt đối hiện tượng sụp đổ nhãn).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from collections import Counter
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Data Augmentation tăng cường đa dạng cho ảnh siêu âm
train_tf = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/content/dataset_multimodal_crops_7cls/train', transform=train_tf)
val_ds = datasets.ImageFolder('/content/dataset_multimodal_crops_7cls/val', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

print(f"Tập huấn luyện 7 lớp: {len(train_ds)} ảnh crops")
print(f"Tập kiểm định 7 lớp: {len(val_ds)} ảnh crops")
print(f"Thứ tự 7 lớp bệnh học: {train_ds.classes}")
assert len(train_ds.classes) == 7, f'Lỗi: Số lượng lớp phải đúng bằng 7, hiện tại là {len(train_ds.classes)}'

# 2. Tính trọng số alpha Class Weights bù đắp mẫu hiếm
counts = Counter(train_ds.targets)
total_samples = len(train_ds)
class_weights = [total_samples / (len(counts) * counts[i]) for i in range(len(counts))]
weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"Trọng số Class Weights (alpha): {[round(w, 2) for w in class_weights]}")

# 3. Định nghĩa Focal Loss (gamma=2.0)
class FocalLoss(nn.Module):
    """
    Focal Loss đa lớp: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Hạ thấp gradient của các mẫu dễ, khuếch đại gradient cho nhóm u hiếm.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

# 4. Khởi tạo EfficientNet-B3 pre-trained
weights_b3 = EfficientNet_B3_Weights.DEFAULT
model_b3 = efficientnet_b3(weights=weights_b3)
in_f = model_b3.classifier[1].in_features
model_b3.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_f, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),
    nn.Dropout(p=0.15),
    nn.Linear(512, 7)
)
model_b3 = model_b3.to(device)

criterion = FocalLoss(alpha=weights_tensor, gamma=2.0)
optimizer = torch.optim.AdamW(model_b3.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS_B3 = 20
best_val_macro_f1 = 0.0
best_val_acc = 0.0
save_b3_path = '/content/runs_multimodal/efficientnet_b3_best.pth'
os.makedirs('/content/runs_multimodal', exist_ok=True)

print("\n🚀 Bắt đầu huấn luyện EfficientNet-B3 với Focal Loss (gamma=2.0)...")
for epoch in range(1, EPOCHS_B3 + 1):
    model_b3.train()
    t_loss, t_correct = 0.0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model_b3(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_correct += (outs.argmax(1) == lbls).sum().item()
    scheduler.step()
    
    model_b3.eval()
    v_loss, v_correct = 0.0, 0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model_b3(imgs)
            loss = criterion(outs, lbls)
            v_loss += loss.item() * imgs.size(0)
            preds = outs.argmax(1)
            v_correct += (preds == lbls).sum().item()
            val_preds.extend(preds.cpu().numpy().tolist())
            val_targets.extend(lbls.cpu().numpy().tolist())
            
    train_acc = (t_correct / len(train_ds)) * 100
    val_acc = (v_correct / len(val_ds)) * 100
    val_macro_f1 = f1_score(val_targets, val_preds, average='macro') * 100
    
    print(f"Epoch [{epoch:02d}/{EPOCHS_B3}] - Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Macro-F1: {val_macro_f1:.2f}% ⭐")
    
    # Tiêu chuẩn cốt lõi: Lưu mô hình có Val Macro-F1 cao nhất để bảo vệ lớp thiểu số
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        best_val_acc = val_acc
        torch.save(model_b3.state_dict(), save_b3_path)
        print(f"  -> 🎯 Đã cập nhật Checkpoint tốt nhất (Val Macro-F1: {best_val_macro_f1:.2f}% | Acc: {best_val_acc:.2f}%)")
        
print(f"\n✅ Huấn luyện hoàn tất! Kỷ lục: Best Val Macro-F1 = {best_val_macro_f1:.2f}% | Val Acc = {best_val_acc:.2f}%")
print(f"Trọng số lưu tại: {save_b3_path}")

## 6. Đánh giá Đa Phương Thái: Tách Biệt 2D vs 3D CEUS, Ma trận Chuẩn hóa 7x7 & Báo cáo F1
1. **Quy trình Cascade 2 Giai đoạn**: Đánh giá toàn diện trên 569 ảnh test (2D vs 3D CEUS vs Hợp nhất).
2. **Ma trận Nhầm lẫn Chuẩn hóa 7x7**: Đánh giá chi tiết năng lực phân biệt 7 nhóm bệnh học của EfficientNet-B3 (không còn hàng/cột background).
3. **Báo cáo Chuyên khoa (Classification Report)**: Phân tích sâu chỉ số Macro-F1, Precision, Recall cho từng nhóm bệnh.

In [ ]:
from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import display, HTML

# 1. Load trọng số tốt nhất của cả 2 mô hình
best_yolo = YOLO('/content/runs_multimodal/stage1_yolo11_binary/weights/best.pt')
model_b3.load_state_dict(torch.load(save_b3_path))
model_b3.eval()

class_names = train_ds.classes
CONF_SCREEN = 0.30

val_img_dir = Path('/content/dataset_multimodal_binary/images/val')
val_lbl_dir = Path('/content/dataset_multimodal_binary/labels/val')

results_records = []
all_val_imgs = list(val_img_dir.glob('*.JPG'))

print(f"Chạy suy luận Cascaded Pipeline trên {len(all_val_imgs)} ảnh validation...")

for img_p in all_val_imgs:
    fname = img_p.name
    modality = '2D' if fname.startswith('2d_') else '3D_CEUS'
    base_name = img_p.stem
    
    # Ground Truth nhị phân
    lbl_p = val_lbl_dir / f"{base_name}.txt"
    is_gt_tumor = False
    if lbl_p.exists():
        with open(lbl_p, 'r') as lf:
            if len(lf.read().strip()) > 0: is_gt_tumor = True
            
    # GIAI ĐOẠN 1: YOLO11 Sàng lọc
    y_preds = best_yolo.predict(source=str(img_p), conf=CONF_SCREEN, verbose=False)
    has_detected_box = len(y_preds) > 0 and len(y_preds[0].boxes) > 0
    
    pred_disease = False
    pred_class_name = 'Normal_Ovary'
    pred_conf = 0.0
    risk_level = 'Bình thường'
    
    if has_detected_box:
        pred_disease = True
        # GIAI ĐOẠN 2: Cắt ROI với 15% Padding và đưa qua EfficientNet-B3
        box = y_preds[0].boxes.xyxy[0].cpu().numpy().astype(int)
        img_cv = cv2.imread(str(img_p))
        roi = crop_roi(img_cv, box, pad=0.15)
        if roi is not None:
            roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(roi_rgb)
            inp = val_tf(pil_img).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model_b3(inp)
                probs = torch.softmax(logits, dim=1)[0]
                top_idx = probs.argmax().item()
                pred_class_name = class_names[top_idx]
                pred_conf = probs[top_idx].item() * 100
                
            # Phân tầng nguy cơ ác tính
            if pred_class_name == 'High_grade_serous' or probs[-1].item() >= 0.30:
                risk_level = 'Ác tính (High-grade)'
            else:
                risk_level = 'Lành tính / U nang'
                
    results_records.append({
        'filename': fname,
        'modality': modality,
        'gt_tumor': is_gt_tumor,
        'pred_tumor': pred_disease,
        'pred_class': pred_class_name,
        'confidence': pred_conf,
        'risk_level': risk_level
    })

df_res = pd.DataFrame(results_records)

# Hàm tính toán chỉ số theo từng nhóm
def calc_stats(sub_df, name):
    tp = len(sub_df[(sub_df['gt_tumor'] == True) & (sub_df['pred_tumor'] == True)])
    fp = len(sub_df[(sub_df['gt_tumor'] == False) & (sub_df['pred_tumor'] == True)])
    tn = len(sub_df[(sub_df['gt_tumor'] == False) & (sub_df['pred_tumor'] == False)])
    fn = len(sub_df[(sub_df['gt_tumor'] == True) & (sub_df['pred_tumor'] == False)])
    total = len(sub_df)
    sens = (tp / (tp + fn)) * 100 if (tp + fn) > 0 else 0
    spec = (tn / (tn + fp)) * 100 if (tn + fp) > 0 else 0
    acc = ((tp + tn) / total) * 100 if total > 0 else 0
    return {
        'Bộ Dữ Liệu': name,
        'Số lượng Mẫu': total,
        'Đoán Đúng U (TP)': tp,
        'Báo Động Giả (FP)': fp,
        'Chuẩn Âm Tính (TN)': tn,
        'Bỏ Sót U (FN)': fn,
        'Độ Nhạy (Sensitivity)': f"{sens:.2f}%",
        'Độ Đặc Hiệu (Specificity)': f"{spec:.2f}%",
        'Độ Chính Xác Sàng Lọc': f"{acc:.2f}% ⭐"
    }

summary_data = [
    calc_stats(df_res[df_res['modality'] == '2D'], 'Siêu âm 2D Thông thường'),
    calc_stats(df_res[df_res['modality'] == '3D_CEUS'], 'Siêu âm 3D CEUS Cản âm vi mạch'),
    calc_stats(df_res, 'TỔNG HỢP TOÀN BỘ (2D + 3D CEUS)')
]
df_summary = pd.DataFrame(summary_data)

html_markup = f"""
<div style='font-family: Arial, sans-serif; max-width: 950px; margin: 20px 0;'>
  <h3 style='color: #0284c7; border-bottom: 2px solid #0284c7; padding-bottom: 8px;'>🏆 BẢNG TỔNG HỢP HIỆU NĂNG ĐA PHƯƠNG THÁI (2D vs 3D CEUS vs HỢP NHẤT)</h3>
  {df_summary.to_html(index=False, classes='table table-bordered table-striped')}
</div>
"""
display(HTML(html_markup))
df_summary.to_csv('/content/runs_multimodal/multimodal_summary_report.csv', index=False, encoding='utf-8-sig')

# ==========================================================================
# 2. MA TRẬN NHẦM LẪN CHUẨN HÓA 7x7 & CLASSIFICATION REPORT (STAGE 2)
# ==========================================================================
print("\n" + "=" * 65)
print("📊 ĐÁNH GIÁ CHI TIẾT GIAI ĐOẠN 2: EFFICIENTNET-B3 TRÊN 7 NHÓM BỆNH HỌC")
print("=" * 65)

y_true_7, y_pred_7 = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        imgs = imgs.to(device)
        outs = model_b3(imgs)
        y_pred_7.extend(outs.argmax(1).cpu().numpy().tolist())
        y_true_7.extend(lbls.numpy().tolist())

# Báo cáo Y khoa chi tiết
clf_report = classification_report(y_true_7, y_pred_7, target_names=class_names, digits=4)
print(clf_report)

# Vẽ Ma trận nhầm lẫn chuẩn hóa 7x7 sạch bóng background
cm_norm = confusion_matrix(y_true_7, y_pred_7, normalize='true')

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm, 
    annot=True, 
    fmt='.2f', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar=True
)
plt.title('Ma trận Nhầm lẫn Chuẩn hóa 7 Nhóm U Buồng trứng (EfficientNet-B3)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Nhãn Dự Đoán (Predicted Label)', fontsize=11, fontweight='bold')
plt.ylabel('Nhãn Thực Tế (True Label)', fontsize=11, fontweight='bold')
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

cm_save_path = '/content/runs_multimodal/stage2_confusion_matrix_7cls_normalized.png'
plt.savefig(cm_save_path, dpi=300)
plt.show()
print(f"✅ Đã lưu Ma trận nhầm lẫn 7x7 chuẩn Y khoa: {cm_save_path}")

## 7. Explainable AI: Bản Đồ Nhiệt Grad-CAM & Phân Tầng Rủi Ro Lâm Sàng
- **Tiêu chuẩn công bố Quốc tế**: Áp dụng kỹ thuật **Grad-CAM (Gradient-weighted Class Activation Mapping)** trực tiếp trên tầng tích chập cuối `features[-1]` của EfficientNet-B3.
- **Mục tiêu kiểm chứng lâm sàng**:
  - **Mô hình học ĐÚNG**: Bản đồ nhiệt đỏ rực tập trung tại vách ngăn dày, chồi sùi đặc bên trong nang hoặc bờ viền xâm lấn ra mô lành.
  - **Mô hình học SAI**: Bản đồ nhiệt tập trung ở góc đen ngoài rìa, vệt thước đo hoặc chữ số kỹ thuật của máy siêu âm.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

# ==========================================================================
# MODULE GRAD-CAM THUẦN PYTORCH (ZERO-DEPENDENCY)
# ==========================================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        self.hook_f = target_layer.register_forward_hook(self._save_activation)
        self.hook_b = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
            
        self.model.zero_grad()
        loss = output[0, target_class]
        loss.backward()

        # Global Average Pooling trên gradients để xác định trọng số từng kênh
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1).squeeze()
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.cpu().numpy(), target_class

# Khởi tạo Grad-CAM trên tầng tích chập cuối cùng của EfficientNet-B3
cam_extractor = GradCAM(model_b3, model_b3.features[-1])

# Chọn 4 ca bệnh kiểm định đại diện
sample_test_imgs = all_val_imgs[:4]
fig, axes = plt.subplots(len(sample_test_imgs), 3, figsize=(15, 4 * len(sample_test_imgs)))

for i, p in enumerate(sample_test_imgs):
    row = df_res[df_res['filename'] == p.name].iloc[0]
    img_bgr = cv2.imread(str(p))
    y_preds = best_yolo.predict(source=str(p), conf=CONF_SCREEN, verbose=False)
    
    roi_img = None
    overlay = None
    cam_title = "Không phát hiện tổn thương"
    
    if len(y_preds) > 0 and len(y_preds[0].boxes) > 0:
        box = y_preds[0].boxes.xyxy[0].cpu().numpy().astype(int)
        roi_bgr = crop_roi(img_bgr, box, pad=0.15)
        
        if roi_bgr is not None:
            roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(roi_rgb)
            inp = val_tf(pil_img).unsqueeze(0).to(device)
            
            # Tạo Heatmap Grad-CAM
            cam_map, pred_c = cam_extractor.generate(inp)
            cam_resized = cv2.resize(cam_map, (300, 300))
            heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
            heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            
            # Hòa trộn với ảnh ROI gốc (alpha blending)
            overlay = cv2.addWeighted(roi_rgb, 0.60, heatmap_rgb, 0.40, 0)
            roi_img = roi_rgb
            cam_title = f"Vùng tập trung chú ý: {class_names[pred_c]}"
            
        # Vẽ Bounding Box trên ảnh toàn cảnh
        color = (0, 0, 255) if 'Ác tính' in row['risk_level'] else (0, 255, 0)
        cv2.rectangle(img_bgr, (box[0], box[1]), (box[2], box[3]), color, 3)
        txt = f"{row['pred_class']} ({row['confidence']:.1f}%)"
        cv2.putText(img_bgr, txt, (box[0], max(25, box[1] - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
        
    img_full_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Cột 1: Toàn cảnh siêu âm & Bounding Box YOLO11
    axes[i, 0].imshow(img_full_rgb)
    axes[i, 0].axis('off')
    axes[i, 0].set_title(f"[{row['modality']}] Định vị Khối U\n{row['pred_class']} | {row['risk_level']}", fontsize=10, fontweight='bold')
    
    # Cột 2: Cắt ROI cận cảnh (Padding 15%)
    if roi_img is not None:
        axes[i, 1].imshow(roi_img)
    else:
        axes[i, 1].imshow(img_full_rgb)
    axes[i, 1].axis('off')
    axes[i, 1].set_title("Ảnh Cắt ROI (15% Biên Padding)", fontsize=10, fontweight='bold')
    
    # Cột 3: Bản đồ nhiệt Grad-CAM Explainable AI
    if overlay is not None:
        axes[i, 2].imshow(overlay)
    else:
        axes[i, 2].imshow(img_full_rgb)
    axes[i, 2].axis('off')
    axes[i, 2].set_title(f"Explainable AI (Grad-CAM)\n{cam_title}", fontsize=10, fontweight='bold')

plt.tight_layout()
gradcam_save_path = '/content/runs_multimodal/gradcam_explainable_ai.png'
plt.savefig(gradcam_save_path, dpi=300)
plt.show()
print(f"✅ Đã lưu ảnh minh chứng Explainable AI (Grad-CAM): {gradcam_save_path}")

## 8. Tự động Sao lưu Toàn bộ Trọng số & Kết quả về Google Drive
Lưu các file quan trọng vào thư mục `MyDrive/NCKH_Multimodal_Results/`.

In [ ]:
backup_dir = Path('/content/drive/MyDrive/NCKH_Multimodal_Results')
os.makedirs(backup_dir, exist_ok=True)

# 1. Sao lưu Model 1 YOLO11 Binary
yolo_best = '/content/runs_multimodal/stage1_yolo11_binary/weights/best.pt'
if os.path.exists(yolo_best):
    shutil.copy2(yolo_best, backup_dir / 'stage1_yolo11_binary_multimodal_best.pt')
    print(f"✅ Đã sao lưu YOLO11 Binary: {backup_dir}/stage1_yolo11_binary_multimodal_best.pt")

# 2. Sao lưu Model 2 EfficientNet-B3 (Tối ưu bởi Focal Loss & Macro-F1)
if os.path.exists(save_b3_path):
    shutil.copy2(save_b3_path, backup_dir / 'stage2_efficientnet_b3_multimodal_best.pth')
    print(f"✅ Đã sao lưu EfficientNet-B3: {backup_dir}/stage2_efficientnet_b3_multimodal_best.pth")

# 3. Sao lưu Báo cáo tổng hợp CSV
sum_csv = '/content/runs_multimodal/multimodal_summary_report.csv'
if os.path.exists(sum_csv):
    shutil.copy2(sum_csv, backup_dir / 'multimodal_summary_report.csv')
    print(f"✅ Đã sao lưu Báo cáo CSV: {backup_dir}/multimodal_summary_report.csv")

# 4. Sao lưu Ma trận nhầm lẫn 7x7 sạch bóng background
if os.path.exists(cm_save_path):
    shutil.copy2(cm_save_path, backup_dir / 'stage2_confusion_matrix_7cls_normalized.png')
    print(f"✅ Đã sao lưu Ma trận nhầm lẫn 7x7: {backup_dir}/stage2_confusion_matrix_7cls_normalized.png")

# 5. Sao lưu Ảnh phân tích Explainable AI Grad-CAM
if os.path.exists(gradcam_save_path):
    shutil.copy2(gradcam_save_path, backup_dir / 'gradcam_explainable_ai.png')
    print(f"✅ Đã sao lưu Ảnh Grad-CAM: {backup_dir}/gradcam_explainable_ai.png")

print(f"\n🎉 HOÀN TẤT TOÀN DIỆN! Toàn bộ mô hình, ma trận 7x7, báo cáo và ảnh Grad-CAM đã được bảo toàn trên Google Drive!")